# setup_wizard

**Source:** `05_orchestration/setup_wizard.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Complete automated setup wizard for Databricks ingestion framework."""

from __future__ import annotations

import sys
from pathlib import Path


## Section 2: Define `_resolve_common_dir()` function with logic for processing

This cell handles: *Define `_resolve_common_dir()` function with logic for processing*


In [ ]:
def _resolve_common_dir() -> Path:
    """Resolve notebooks/00_common path in both local and Databricks runtimes."""
    candidates: list[Path] = []

    # File/module execution path (local, pytest, script execution)
    file_path = globals().get("__file__")
    if file_path:
        current_dir = Path(file_path).resolve().parent
        candidates.extend([
            current_dir.parent / "00_common",
            current_dir / "00_common",
        ])

    # Notebook/runtime cwd fallbacks
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd / "notebooks" / "00_common",
        cwd / "00_common",
        cwd.parent / "00_common",
    ])

    # Databricks notebook path fallback
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()  # type: ignore[name-defined]
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        nb_path = Path(str(notebook_path))
        candidates.extend([
            nb_path.parent.parent / "00_common",
            Path("/Workspace") / nb_path.parent.parent / "00_common",
        ])
    except Exception:
        pass

    # Preserve order, avoid duplicate checks
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists() and candidate.is_dir():
            return candidate

    tried = "\n".join(f"- {p}" for p in candidates)
    raise ModuleNotFoundError(
        "Unable to locate notebooks/00_common for imports. Tried:\n"
        f"{tried}"
    )


COMMON_DIR = _resolve_common_dir()
if str(COMMON_DIR) not in sys.path:
    sys.path.append(str(COMMON_DIR))

try:
    from global_config import load_global_config
    from utils import get_logger
except ModuleNotFoundError:
    # Databricks notebook-only deployments may not expose .py modules on sys.path.
    import json
    import logging
    import os
    import re
    import subprocess
    from datetime import datetime, timezone

    try:
        import yaml
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "PyYAML"])
        import yaml

    _ENV_PATTERN = re.compile(r"\$\{([A-Z0-9_]+)(?::([^}]*))?\}")

    def _substitute_env(value: str) -> str:
        def replacer(match: re.Match[str]) -> str:
            key = match.group(1)
            default_value = match.group(2) if match.group(2) is not None else ""
            return os.getenv(key, default_value)

        return _ENV_PATTERN.sub(replacer, value)

    def _resolve_obj(obj):
        if isinstance(obj, str):
            return _substitute_env(obj)
        if isinstance(obj, dict):
            return {k: _resolve_obj(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_resolve_obj(v) for v in obj]
        return obj

    def load_global_config(config_path: str) -> dict:
        path = Path(config_path)
        if not path.exists():
            raise FileNotFoundError(
                f"Global config not found: {path}. "
                "Set --global-config to the correct path or check your deployment."
            )
        with path.open("r", encoding="utf-8") as f:
            payload = yaml.safe_load(f) or {}
        if not isinstance(payload, dict):
            raise ValueError(f"Global config root must be a mapping/object: {path}")
        if "lifecycle_log_table" not in payload:
            dbx = payload.get("databricks", {}) if isinstance(payload, dict) else {}
            catalog = dbx.get("catalog", "")
            audit_schema = dbx.get("audit_schema", "audit")
            if catalog:
                payload["lifecycle_log_table"] = f"{catalog}.{audit_schema}.entity_lifecycle_log"
            else:
                payload["lifecycle_log_table"] = f"{audit_schema}.entity_lifecycle_log"
        resolved = _resolve_obj(payload)
        if not isinstance(resolved, dict):
            raise ValueError(f"Global config resolution produced non-mapping object: {path}")
        return resolved

    class _JsonFormatter(logging.Formatter):
        def format(self, record: logging.LogRecord) -> str:
            payload = {
                "ts": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(),
                "level": record.levelname,
                "logger": record.name,
                "message": record.getMessage(),
            }
            if record.exc_info:
                payload["exc_info"] = self.formatException(record.exc_info)
            return json.dumps(payload, default=str)

    def get_logger(name: str = "ingestion_framework") -> logging.Logger:
        logger = logging.getLogger(name)
        if logger.handlers:
            return logger
        logger.setLevel(logging.DEBUG)
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(_JsonFormatter())
        logger.addHandler(handler)
        logger.propagate = False
        return logger

_logger = get_logger("framework_setup_wizard")


## Section 3: Define `_step_1_verify_cluster()` function with logic for processing

This cell handles: *Define `_step_1_verify_cluster()` function with logic for processing*


In [ ]:
def _step_1_verify_cluster(spark) -> bool:
    """Verify Spark session is active."""
    try:
        version = spark.version
        _logger.info(f"✓ Spark cluster active (version {version})")
        return True
    except Exception as e:
        _logger.error(f"✗ No Spark cluster: {e}")
        return False


## Section 4: Define `_step_2_verify_libraries()` function with logic for processing

This cell handles: *Define `_step_2_verify_libraries()` function with logic for processing*


In [ ]:
def _step_2_verify_libraries(spark) -> bool:
    """Verify required Python packages are available."""
    required_packages = {
        "pyspark": "PySpark",
        "yaml": "PyYAML",
        "requests": "requests",
    }
    
    all_available = True
    for import_name, display_name in required_packages.items():
        try:
            __import__(import_name)
            _logger.info(f"✓ {display_name} available")
        except ImportError:
            _logger.error(f"✗ {display_name} not found - install via Cluster → Libraries")
            all_available = False
    
    return all_available


## Section 5: Define `_step_3_initialize_uc()` function with logic for processing

This cell handles: *Define `_step_3_initialize_uc()` function with logic for processing*


In [ ]:
def _step_3_initialize_uc(config_path: str, spark) -> bool:
    """Validate UC infrastructure (already initialized by initialize_framework job)."""
    try:
        # Just verify tables exist, don't create them
        # (initialize_framework job handles creation)
        config = load_global_config(config_path)
        dbx_config = config.get("databricks", {})
        catalog = dbx_config.get("catalog")
        control_schema = dbx_config.get("control_schema", "control")
        
        # Check if control tables exist
        try:
            spark.sql(f"DESCRIBE TABLE {catalog}.{control_schema}.source_registry")
            _logger.info("✓ UC infrastructure validated (source_registry exists)")
            return True
        except Exception:
            _logger.error(f"✗ UC tables not found in {catalog}.{control_schema}")
            return False
    except Exception as e:
        _logger.error(f"✗ UC validation error: {e}")
        return False


## Section 6: Define `_step_4_create_sample_tables()` function with logic for processing

This cell handles: *Define `_step_4_create_sample_tables()` function with logic for processing*


In [ ]:
def _step_4_create_sample_tables(spark, catalog: str, bronze_schema: str, silver_schema: str) -> bool:
    """Create sample landing/conformance/silver tables for demo sources."""
    try:
        # Sample table for connect.cemc.countryriskdet
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {catalog}.{bronze_schema}.connect_countryriskdet_landing (
                source_system STRING,
                source_entity STRING,
                load_id STRING,
                ingest_ts TIMESTAMP,
                raw_payload STRING
            )
            USING DELTA
            COMMENT 'Landing table for connect.cemc.countryriskdet'
        """)
        _logger.info(f"✓ Created {catalog}.{bronze_schema}.connect_countryriskdet_landing")

        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {catalog}.{bronze_schema}.connect_countryriskdet_conformance (
                record_id STRING,
                country_code STRING,
                risk_level STRING,
                load_id STRING,
                ingest_ts TIMESTAMP
            )
            USING DELTA
            COMMENT 'Conformance table for connect.cemc.countryriskdet'
        """)
        _logger.info(f"✓ Created {catalog}.{bronze_schema}.connect_countryriskdet_conformance")

        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.connect_countryriskdet (
                record_id STRING,
                country_code STRING,
                risk_level STRING,
                load_id STRING,
                ingest_ts TIMESTAMP,
                processed_ts TIMESTAMP
            )
            USING DELTA
            COMMENT 'Silver table for connect.cemc.countryriskdet'
        """)
        _logger.info(f"✓ Created {catalog}.{silver_schema}.connect_countryriskdet")

        return True
    except Exception as e:
        _logger.error(f"✗ Sample table creation failed: {e}")
        return False


## Section 7: Define `_step_5_load_sample_metadata()` function with logic for processing

This cell handles: *Define `_step_5_load_sample_metadata()` function with logic for processing*


In [ ]:
def _step_5_load_sample_metadata(spark, catalog: str, control_schema: str) -> bool:
    """Load sample metadata into control tables."""
    try:
        # Sample source registry row
        spark.sql(f"""
            INSERT INTO {catalog}.{control_schema}.source_registry (
                tenant, brand, region, product_area, product_name, source_system, source_entity,
                source_type, source_format, source_path,
                jdbc_profile, jdbc_table, api_profile, api_endpoint, api_method, api_query_params_json,
                load_type, watermark_column,
                landing_table, conformance_table, silver_table,
                primary_key, publish_mode, is_active,
                source_options_json,
                pre_landing_transform_notebook, post_conformance_transform_notebook, custom_publish_notebook,
                scheduler_name, schedule_cron, retention_days, sttm_profile,
                landing_table_type, landing_table_path, conformance_table_type, conformance_table_path,
                silver_table_type, silver_table_path
            ) VALUES (
                'ikea', 'ikea', 'eu', 'risk', 'connect', 'cemc', 'countryriskdet',
                'FILE', 'json_jsonl_mixed', '${RAW_CONNECT_COUNTRYRISKDET_ROOT:abfss://raw@<storage-account>.dfs.core.windows.net/connect/cemccountryriskdet}',
                NULL, NULL, NULL, NULL, NULL, NULL,
                'incremental', 'file_modification_ts',
                '{catalog}.{control_schema}.bronze.connect_countryriskdet_landing',
                '{catalog}.{control_schema}.bronze.connect_countryriskdet_conformance',
                '{catalog}.{control_schema}.silver.connect_countryriskdet',
                'record_id', 'append', true,
                '{{"file_ingest_mode":"batch","recursiveFileLookup":"true"}}',
                NULL, NULL, NULL,
                NULL, NULL, NULL, NULL,
                'managed', NULL, 'managed', NULL, 'managed', NULL
            )
        """)
        _logger.info(f"✓ Loaded sample source registry row")
        return True
    except Exception as e:
        _logger.error(f"✗ Metadata load failed (may already exist): {e}")
        # Not a failure - table may already have data
        return True


## Section 8: Define `_step_6_verify_config()` function with logic for processing

This cell handles: *Define `_step_6_verify_config()` function with logic for processing*


In [ ]:
def _step_6_verify_config(config_path: str) -> dict:
    """Load and verify global config."""
    try:
        cfg = load_global_config(config_path)
        db_cfg = cfg.get("databricks", {})
        
        _logger.info(f"✓ Config loaded:")
        _logger.info(f"  - Catalog: {db_cfg.get('catalog')}")
        _logger.info(f"  - Bronze Schema: {db_cfg.get('bronze_schema')}")
        _logger.info(f"  - Silver Schema: {db_cfg.get('silver_schema')}")
        _logger.info(f"  - Audit Schema: {db_cfg.get('audit_schema')}")
        _logger.info(f"  - Metadata Mode: {cfg.get('metadata', {}).get('mode')}")
        
        return cfg
    except Exception as e:
        _logger.error(f"✗ Config verification failed: {e}")
        return {}


## Section 9: Define `_step_7_test_orchestrator()` function with logic for processing

This cell handles: *Define `_step_7_test_orchestrator()` function with logic for processing*


In [ ]:
def _step_7_test_orchestrator(spark, config: dict, config_dir: str) -> bool:
    """Run a dry-run test of the orchestrator."""
    try:
        from framework_orchestrator import run_all
        
        _logger.info("Running orchestrator dry-run test...")
        results = run_all(
            config_dir=config_dir,
            global_config=config,
            spark=spark,
            dry_run=True,
            product_name="connect",
            source_system="cemc",
            source_entity="countryriskdet",
            pk_check_summary=True
        )
        
        if results:
            _logger.info(f"✓ Orchestrator dry-run successful: {len(results)} source(s) processed")
            import json
            _logger.info(f"  Result: {json.dumps(results[0], indent=2)[:200]}...")
            return True
        else:
            _logger.warning("⚠ Orchestrator returned no results (check source_registry.is_active)")
            return True
    except Exception as e:
        _logger.error(f"✗ Orchestrator test failed: {e}")
        return False


## Section 10: Define `run_complete_setup()` function with logic for processing

This cell handles: *Define `run_complete_setup()` function with logic for processing*


In [ ]:
def run_complete_setup(config_path: str) -> bool:
    """Run complete automated setup wizard."""
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.getActiveSession()
        if spark is None:
            _logger.error("✗ No active Spark session. Run this in Databricks only.")
            return False
    except ImportError:
        _logger.error("✗ PySpark not available. Run this in Databricks only.")
        return False

    _logger.info("=" * 80)
    _logger.info("DATABRICKS INGESTION FRAMEWORK - AUTOMATED SETUP WIZARD")
    _logger.info("=" * 80)

    config_dir = str(Path(config_path).parent)

    # Step 1: Verify cluster
    _logger.info("\n[1/7] Verifying Spark cluster...")
    if not _step_1_verify_cluster(spark):
        return False

    # Step 2: Verify libraries
    _logger.info("\n[2/7] Verifying required libraries...")
    if not _step_2_verify_libraries(spark):
        _logger.warning("⚠ Some libraries missing - install via Cluster → Libraries")

    # Step 3: Initialize UC
    _logger.info("\n[3/7] Initializing Unity Catalog infrastructure...")
    if not _step_3_initialize_uc(config_path, spark):
        return False

    # Step 4: Load config to get schema names
    _logger.info("\n[4/7] Loading configuration...")
    config = _step_6_verify_config(config_path)
    if not config:
        return False

    db_cfg = config.get("databricks", {})
    catalog = db_cfg.get("catalog")
    bronze_schema = db_cfg.get("bronze_schema")
    silver_schema = db_cfg.get("silver_schema")
    control_schema = db_cfg.get("control_schema", "control")

    # Step 5: Create sample tables
    _logger.info("\n[5/7] Creating sample landing/conformance/silver tables...")
    if not _step_4_create_sample_tables(spark, catalog, bronze_schema, silver_schema):
        _logger.warning("⚠ Sample table creation had issues - may already exist")

    # Step 6: Load sample metadata
    _logger.info("\n[6/7] Loading sample metadata...")
    _step_5_load_sample_metadata(spark, catalog, control_schema)

    # Step 7: Test orchestrator
    _logger.info("\n[7/7] Testing orchestrator with dry-run...")
    _step_7_test_orchestrator(spark, config, config_dir)

    _logger.info("\n" + "=" * 80)
    _logger.info("✓ SETUP COMPLETE!")
    _logger.info("=" * 80)
    _logger.info("\nNext steps:")
    _logger.info("1. Review sample metadata in control tables")
    _logger.info("2. Add your own sources to config/source_registry.csv")
    _logger.info("3. Configure column mappings in config/column_mapping.csv")
    _logger.info("4. Define DQ rules in config/dq_rules.csv")
    _logger.info("5. Run orchestrator in execute mode: framework_orchestrator.py --execute")
    _logger.info("\nDocumentation:")
    _logger.info("  - Setup Guide: docs/DATABRICKS_QUICKSTART.md")
    _logger.info("  - Framework Overview: README.md")
    _logger.info("  - Architecture: docs/architecture.md")
    _logger.info("=" * 80)

    return True


## Section 11: Define `_resolve_default_global_config_path()` function with logic for processing

This cell handles: *Define `_resolve_default_global_config_path()` function with logic for processing*


In [ ]:
def _resolve_default_global_config_path() -> str:
    """Resolve config/global_config.yaml across script and notebook runtimes."""
    candidates: list[Path] = []

    # Script/module execution path
    file_path = globals().get("__file__")
    if file_path:
        base = Path(file_path).resolve()
        candidates.extend([
            base.parents[2] / "config" / "global_config.yaml",
            base.parent / "config" / "global_config.yaml",
        ])

    # Notebook/local cwd fallbacks
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd / "config" / "global_config.yaml",
        cwd.parent / "config" / "global_config.yaml",
        cwd.parent.parent / "config" / "global_config.yaml",
    ])

    # Databricks workspace notebook path fallback
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()  # type: ignore[name-defined]
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        nb_path = Path(str(notebook_path))
        workspace_base = Path("/Workspace") / nb_path.parent.parent.parent
        candidates.append(workspace_base / "config" / "global_config.yaml")
    except Exception:
        pass

    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists() and candidate.is_file():
            return str(candidate)

    # Keep a sensible default even if not found; downstream validation gives clear error.
    return str((Path.cwd().resolve() / "config" / "global_config.yaml"))


## Section 12: Define `_resolve_config_path_from_widget_or_args()` function with logic for processing

This cell handles: *Define `_resolve_config_path_from_widget_or_args()` function with logic for processing*


In [ ]:
def _resolve_config_path_from_widget_or_args(arg_config: str | None) -> str:
    """Resolve global config path from Databricks widget, CLI arg, or runtime defaults."""
    # 1) Databricks notebook task parameter (highest priority in notebook runs)
    widget_value: str | None = None
    try:
        dbutils.widgets.text("global-config", "")  # type: ignore[name-defined]
        raw = dbutils.widgets.get("global-config")  # type: ignore[name-defined]
        widget_value = str(raw).strip() or None
    except Exception:
        widget_value = None

    # 2) CLI arg
    arg_value = str(arg_config).strip() if arg_config else None

    # 3) fallback discovery
    chosen = widget_value or arg_value or _resolve_default_global_config_path()

    # Normalize and return exactly what the caller should use.
    return str(Path(chosen).as_posix())


## Section 13: Define `main()` function with logic for processing

This cell handles: *Define `main()` function with logic for processing*


In [ ]:
def main() -> int:
    """Entry point. Returns process exit code."""
    import argparse

    parser = argparse.ArgumentParser(description="Complete automated setup wizard for ingestion framework")
    parser.add_argument("--global-config", default=None)
    # Notebook kernels inject extra argv values; ignore unknown args there.
    args, _unknown = parser.parse_known_args()

    config_path = _resolve_config_path_from_widget_or_args(args.global_config)
    _logger.info(f"Using global config path: {config_path}")

    success = run_complete_setup(config_path)
    return 0 if success else 1


if __name__ == "__main__":
    in_notebook = "ipykernel" in sys.modules and "__file__" not in globals()
    exit_code = main()
    if in_notebook:
        if exit_code != 0:
            raise RuntimeError("Setup wizard failed. See task logs for details.")
    else:
        raise SystemExit(exit_code)
